In [23]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
import warnings
warnings.filterwarnings('ignore')
import joblib

In [2]:
df=pd.read_csv('../artifacts/clean_data.csv')

In [3]:
df.head()

,Unnamed: 0,age,bmi,children,charges,sex_male,smoker_yes,region_northwest,region_southeast,region_southwest
0,0,19,27.900,0,16884.92400,0,1,0,0,1
1,1,18,33.770,1,1725.55230,1,0,0,1,0
2,2,28,33.000,3,4449.46200,1,0,0,1,0
3,3,33,22.705,0,21984.47061,1,0,1,0,0
4,4,32,28.880,0,3866.85520,1,0,1,0,0


In [4]:
X=df.drop(['charges','Unnamed: 0'], axis=1)
y=df['charges']

In [5]:

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [6]:
scaled=StandardScaler()
X_train_scale=scaled.fit_transform(X_train)
X_test_scale=scaled.fit_transform(X_test)

In [7]:
model=KNeighborsRegressor(n_neighbors=15)

In [8]:
fit_model=model.fit(X_train_scale, y_train)

In [9]:
fit_model

,n_neighbors,15
,weights,'uniform'
,algorithm,'auto'
,leaf_size,30
,p,2
,metric,'minkowski'
,metric_params,None
,n_jobs,None


In [10]:
y_pre=fit_model.predict(X_test_scale)

In [11]:
r2=r2_score(y_test, y_pre)

In [12]:
print(r2)

0.5210160343284921


In [13]:
models={
    "linear_Regression":LinearRegression(),
    "KNN":KNeighborsRegressor(),
    "svr":SVR(),
    "decision_tree":DecisionTreeRegressor(),
    "random_Forest":RandomForestRegressor(),
    "xgboost_regression":XGBRegressor()
}

In [14]:
result=[]

In [15]:
params = {
    "linear_Regression":{
        'fit_intercept':[True , False], 
        "copy_X":[True, False],
        "n_jobs":[None , 1 , 2 , 3 , 5 , 9 , 25]
    },
    "KNN":{
        "n_neighbors":[3 , 5 , 7 , 9 , 13 , 25],
        "weights":['uniform' , 'distance'],
        "algorithm":['auto', 'ball_tree', 'kd_tree', 'brute'],
        'leaf_size':[10, 20 , 30 , 35 , 40],
    },
    "svr":{
        "kernel":['linear', 'poly', 'rbf', 'sigmoid'],
        "degree":[1 , 3 , 6 , 7 , 9 , 13, 25],
        "gamma":['scale', 'auto'],
        "epsilon":[0.1,0.3,0.5,0.6,0.8,0.9,1]
    },
    "decision_tree":{
        "criterion": ["squared_error", "absolute_error", "friedman_mse"],
        "max_depth": [None, 3, 5, 10, 20, 30],
        "min_samples_split":[2,3,4,5,6,7,8,9,10],
        "min_samples_leaf":[2,3,4,5,6,7,8,9,10],
        "max_features": [1 , "sqrt", "log2", None],
    },
    "random_Forest":{
        "criterion":['squared_error', 'absolute_error', 'poisson'],
        "max_depth":[None , 1 , 2 , 5 , 7 , 13 , 25],
        "min_samples_split":[2, 3 , 5],
        "min_samples_leaf":[1 , 2 , 3 , 5 , 7 , 9 , 25]
    },
    "xgboost_regression":{
        'n_estimators': [10,50,100,150,250],         # Number of gradient boosted trees
    'learning_rate': [0.1,0.2,0.3,0.4],       # Step size shrinkage (eta)
    'max_depth': [3,4,5,6,7,8,9,10],                # Maximum depth of a tree
    
    # Overfitting Control & Minimum Loss Reduction
    'min_child_weight': [1,2,3,4,5,6,7,8,9,10],         # Minimum sum of instance weight needed in a child
    'gamma': [0.01,0.1,0.2,0.3,0.5],                   # Minimum loss reduction required to make a split
    
    # Stochastic Sampling & Robustness
    'subsample': [0.01,0.1,0.2,0.3,0.5],             # Subsample ratio of the training instances
    'colsample_bytree':[0.01,0.1,0.2,0.3,0.5],      # Subsample ratio of columns when constructing each tree
    
    # Regularization
    'reg_alpha':[0.01,0.1,0.2,0.3,0.5],               # L1 regularization term on weights (Lasso)
    'reg_lambda': [0.01,0.1,0.2,0.3,0.5]            # L2 regularization term on weights (Ridge)
    }
}

In [16]:
best_model=None
best_model_name=None
best_model_score=0

In [17]:
# for name, model in models.items():

#     print('-'*100)
#     print('name:- ', name)
#     search_cv=RandomizedSearchCV(estimator=model,cv=5, param_distributions=params[name], n_iter=20, random_state=42, scoring='r2')
#     model_fit=search_cv.fit(X_train_scale, y_train)
#     y_pred=model_fit.predict(X_test_scale)
    

In [18]:
for name , model in models.items():
    print('-'*100)
    print('Name ' , name )
    search = RandomizedSearchCV(
    estimator=model ,cv=5, param_distributions=params[name], n_iter=20 ,random_state=42 ,scoring="r2")
    model = search.fit(X_train_scale , y_train)
    y_predict = model.predict(X_test_scale)
    print("Best Score ", model.best_score_)
    print("R2 Score ", r2_score(y_test, y_predict))
    
    print("Best Parameters ", model.best_params_)
    if model.best_score_ > best_model_score:
        best_model = model
        best_model_name = name
        best_model_score = model.best_score_
    print('-'*100)

print('Best Modal Name ', best_model_name)
print('Best Modal Score ', best_model_score)

----------------------------------------------------------------------------------------------------
Name  linear_Regression
Best Score  0.5996119557653584
R2 Score  0.5732635406093913
Best Parameters  {'n_jobs': None, 'fit_intercept': True, 'copy_X': True}
----------------------------------------------------------------------------------------------------
----------------------------------------------------------------------------------------------------
Name  KNN
Best Score  0.5828437402841076
R2 Score  0.5159956529066967
Best Parameters  {'weights': 'uniform', 'n_neighbors': 13, 'leaf_size': 35, 'algorithm': 'ball_tree'}
----------------------------------------------------------------------------------------------------
----------------------------------------------------------------------------------------------------
Name  svr
Best Score  0.04388641467169711
R2 Score  0.026643920036317548
Best Parameters  {'kernel': 'linear', 'gamma': 'auto', 'epsilon': 0.1, 'degree': 6}
---------

In [19]:
y_predict=best_model.predict(X_test_scale)

In [21]:
r2=r2_score(y_test, y_predict)

In [22]:
r2

0.6134366693450594

In [28]:
joblib.dump(best_model, '../models/train_model.pkl')
joblib.dump(scaled, '../models/scaler.pkl')

['../models/scaler.pkl']